# Visualize Analysis Results

This tutorial shows how to inspect common scAtlasPy analysis outputs after
quality control, preprocessing, dimensionality reduction, clustering, marker
analysis, or manual annotation.

The goal is not to document every plotting parameter. Instead, this tutorial
helps you choose an appropriate plot, select the correct data representation,
and interpret the result when working with atlas-scale datasets.

By the end of this tutorial, you will be able to:

- inspect quality-control and feature-selection results;
- examine PCA and UMAP using biological and technical metadata;
- compare marker expression across clusters or annotations;
- choose an appropriate expression field for each plot;
- control the number of cells displayed in large scatter plots;
- save figures for downstream inspection and reporting.

## Before You Begin

This tutorial assumes that an existing Atlas has already been opened:



In [ ]:
import os
from pathlib import Path
import scatlaspy as sap

os.chdir(Path("~/scAtlaspy-code-analysis").expanduser())

atlas_path = Path("./tmp/tutorials/basic_pbmc3k/pbmc3k_basic_copy.sasql")

if not atlas_path.is_file():
    raise FileNotFoundError(f"Atlas database not found: {atlas_path}")

atlas = sap.Atlas(
    atlas_path,
    db_memory_limit="8GB",
)



Different sections require different results from the basic workflow.

| Plot type | Required result |
|---|---|
| Quality-control plots | QC metrics stored in `obs` |
| HVG plot | Highly variable gene statistics stored in `var` |
| PCA plots | PCA coordinates and variance statistics |
| Cluster-colored plots | A clustering column such as `kmeans` in `obs` |
| UMAP plots | UMAP coordinates |
| Marker-expression plots | A suitable expression field and a grouping column |
| Annotation plots | An annotation column such as `cell_type_manual` |

You do not need every result to use this tutorial. Follow the sections that
match the current state of your Atlas.

```{note}
Return to the {doc}`../basic/index` if the required analysis results have not
yet been calculated.
```

## Choose a Plot

Start from the question you want to answer.

| Question | Useful plots |
|---|---|
| Are a small number of genes contributing unusually large fractions of the library? | `sap.pl.highest_expr_genes()` |
| Do the QC distributions support the selected filtering thresholds? | `sap.pl.violin_qc_metrics()` and `sap.pl.scatter_qc_metrics()` |
| Does HVG selection capture the expected mean–variance relationship? | `sap.pl.highly_variable_genes()` |
| How much variation is represented by the leading principal components? | PCA variance-ratio plots |
| How are biological groups and technical covariates distributed in PCA space? | `sap.pl.pca()` |
| How are clusters, samples, batches, or annotations distributed in UMAP space? | `sap.pl.umap()` |
| Do candidate marker genes support the proposed cluster identities? | `sap.pl.dotplot()`, `sap.pl.violin()`, and `sap.pl.stacked_violin()` |

## 1. Inspect Quality-control Results

Examine highly expressed genes before or during quality control:



In [ ]:
sap.pl.highest_expr_genes(
    atlas,
    n_top=20,
)



A small number of mitochondrial, ribosomal, or other highly abundant genes may
account for a large fraction of observed counts. Whether this is expected
depends on the tissue, assay, and biological question.

Inspect the distributions of cell-level QC metrics:



In [ ]:
sap.pl.violin_qc_metrics(
    atlas,
    keys=[
        "n_genes_by_counts",
        "cell_total_counts",
        "pct_counts_mt",
    ],
    sample_n=100000,
)



Examine relationships between QC metrics:



In [ ]:
sap.pl.scatter_qc_metrics(
    atlas,
    sample_n=100000,
)



When reviewing these plots, look for:

- cells in extreme low-count or low-gene tails;
- cells with unusually high library sizes or detected-gene counts;
- cells with elevated mitochondrial fractions;
- sample-specific or batch-specific QC distributions;
- filtering thresholds that remove broad continuous populations rather than
  only extreme observations.

Filtering thresholds should be selected in relation to the dataset rather than
copied mechanically from another analysis.

## 2. Inspect Highly Variable Genes

Visualize the mean–variance relationship used during HVG selection:



In [ ]:
sap.pl.highly_variable_genes(
    atlas,
    flavor="seurat",
)



Use the same HVG flavor that was used when running
`sap.pp.highly_variable_genes()`.

Check that:

- selected genes span an appropriate range of mean expression;
- the selected genes lie above the expected dispersion trend;
- the number of selected genes matches the intended analysis;
- unexpectedly dominant technical or mitochondrial genes have not overwhelmed
  the feature set.

The HVG plot validates the selection procedure, but it does not by itself show
whether the selected genes preserve every biological population of interest.

## 3. Inspect PCA

Examine the fraction of variance represented by each principal component:



In [ ]:
sap.pl.pca_variance_ratio(
    atlas,
    n_pcs=50,
)

sap.pl.pca_variance_ratio_cumsum(
    atlas,
    n_pcs=50,
)



These plots can help identify whether most represented variation is
concentrated in a small number of components or distributed across many
components.

Visualize cells in PCA space:



In [ ]:
sap.pl.pca(
    atlas,
    color="kmeans",
)



Coloring PCA by technical or biological metadata can be more informative than
coloring only by clusters:



In [ ]:
sap.pl.pca(
    atlas,
    color="sample",
)

sap.pl.pca(
    atlas,
    color="n_genes_by_counts",
)



Replace `"sample"` with a metadata column available in your Atlas, such as
`donor`, `batch`, `condition`, `technology`, or `tissue`.

When inspecting PCA, consider:

- whether major biological groups are visible;
- whether a technical covariate dominates one or more components;
- whether QC metrics form strong gradients;
- whether isolated groups may reflect low-quality or unusual samples;
- whether the selected number of principal components captures sufficient
  structure for downstream analysis.

PCA provides a linear view of the data and can reveal broad technical or
biological structure before nonlinear embedding.

## 4. Inspect UMAP

Visualize cluster assignments:



In [ ]:
sap.pl.umap(
    atlas,
    color="kmeans",
    sample_n=100000,
)



Also inspect sample, batch, donor, condition, or other covariates:



In [ ]:
sap.pl.umap(
    atlas,
    color="sample",
    sample_n=100000,
)



After manual annotation, visualize the annotation column:



In [ ]:
sap.pl.umap(
    atlas,
    color="cell_type_manual",
    sample_n=100000,
)


Common expression representations include:

| Representation | Interpretation and typical use |
|---|---|
| `data_count` | Count-scale values imported into the Atlas |
| `data_normalize` | Library-size-normalized values, when available |
| `data_log1p` | Log-transformed normalized expression for marker visualization and group comparisons |
| `data_scale` | Centered-only or centered-and-scaled values from `sap.pp.scale()`, used primarily for PCA or model input |

For most marker-expression plots, use:

In [ ]:
use_data="data_log1p"



For example:



In [ ]:
sap.pl.umap(
    atlas,
    color=["MS4A1", "LYZ", "NKG7"],
    use_data="data_log1p",
    sample_n=100000,
)



```{warning}
Scaled expression values may be negative, clipped, and centered relative to
other cells. They are useful for some computational diagnostics but generally
should not be interpreted as original or normalized expression magnitude.

Use `data_log1p` for routine biological interpretation of marker expression.
```

Before plotting, confirm that the selected field exists and represents the
transformation you intend to communicate.

## 6. Compare Marker Expression Across Groups

Define a set of candidate marker genes:



In [ ]:
marker_genes = [
    "IL7R",
    "CD14",
    "LYZ",
    "MS4A1",
    "CD8A",
    "GNLY",
    "NKG7",
    "PPBP",
]



Summarize expression and detection patterns across clusters:



In [ ]:
sap.pl.dotplot(
    atlas,
    genes=marker_genes,
    groupby="kmeans",
    use_data="data_log1p",
)



A dot plot can be used to compare both expression magnitude and the fraction of
cells expressing each marker, depending on the plotting implementation.

Inspect expression distributions with:



In [ ]:
sap.pl.stacked_violin(
    atlas,
    genes=marker_genes,
    groupby="kmeans",
    use_data="data_log1p",
)



For closer inspection of an individual marker:



In [ ]:
sap.pl.violin(
    atlas,
    genes=["MS4A1"],
    groupby="kmeans",
    use_data="data_log1p",
)



After manual annotation, replace the grouping column:



In [ ]:
sap.pl.dotplot(
    atlas,
    genes=marker_genes,
    groupby="cell_type_manual",
    use_data="data_log1p",
)



When evaluating marker support, consider:

- whether expected positive markers are enriched in the proposed group;
- whether the same markers are also broadly expressed elsewhere;
- whether negative or exclusion markers contradict the annotation;
- whether a cluster contains multiple incompatible marker programs;
- whether marker patterns are consistent across samples or donors.

Cell-type annotation should generally be supported by combinations of markers
rather than a single gene.

## 7. Work with Large Atlases

Displaying every cell is often unnecessary for an initial overview and can make
interactive or static scatter plots slow and visually saturated.

Use `sample_n` where supported. The values in this tutorial are intended for
quick inspection; set `sample_n=None` when you want to draw all cells.




In [ ]:
sap.pl.umap(
    atlas,
    color="kmeans",
    sample_n=100000,
)

sap.pl.pca(
    atlas,
    color="kmeans",
)



Choose `sample_n` based on:

- the total number of cells;
- the number and relative sizes of populations;
- the available plotting memory;
- the purpose of the figure.

```{warning}
A sampled plot may omit or underrepresent rare populations. It may also change
the apparent density of groups.

For rare-cell analysis, increase the sample size, inspect the relevant
population separately, or use a sampling strategy that preserves the groups of
interest.
```

Sampling is most appropriate for scatter-plot visualization. Statistical
summaries and biological conclusions should be calculated from the intended
analysis population rather than inferred only from a displayed subset.

When comparing several plots, use the same sampling configuration whenever
possible so that visual differences are not caused only by different subsets
of cells.

## 8. Save Figures

Create an output directory before saving plots:



In [ ]:
from pathlib import Path

figure_dir = Path("./figures")
figure_dir.mkdir(
    parents=True,
    exist_ok=True,
)



Many plotting functions accept `save_path`:



In [ ]:
sap.pl.umap(
    atlas,
    color="kmeans",
    sample_n=100000,
    save_path=figure_dir / "umap_kmeans.png",
)



Some specialized plotting functions may use a `save` argument instead:



In [ ]:
sap.pl.pca_variance_ratio(
    atlas,
    n_pcs=50,
    save_path="pca_variance_ratio.pdf",
)

sap.pl.pca_variance_ratio_cumsum(
    atlas,
    n_pcs=50,
    save_path="pca_variance_cumsum.png",
)



Consult the corresponding API reference for the saving argument supported by
each plotting function.

When preparing a figure for reporting, also record:

- the Atlas and analysis version;
- the cell population displayed;
- the expression field;
- the sampling size or strategy;
- the grouping and color variables;
- any plotting parameters that affect interpretation.

## Close the Atlas

Close the database connection when this tutorial is complete. This releases
the DuckDB file lock so the same `.sasql` Atlas can be opened by another
notebook or Python session.



In [ ]:
atlas.close()


## Next Steps

Continue with {doc}`query-atlas-with-sql` when you need tabular summaries behind
a plot, such as cluster sizes, sample composition, QC statistics, or marker
expression summaries.

Return to {doc}`resume-existing-atlas` when you need to inspect which analysis
results are available before plotting.